# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge-url/badge.svg)](https://colab.research.google.com/github/pratheekrao12/-flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Lane 2 is **Refresh / Content Opportunity Scoring**. This Week 5 experiment ranks anonymized pages for possible human review. The target is the historical March 16–31 `is_declining_proxy`; it does **not** prove future decline, refresh benefit, or a causal effect.

## 1. Method choice and why

I use **logistic regression** as the first learned model. It produces probabilities that support ranking, its coefficients are relatively interpretable, and its simple linear decision boundary gives a useful honest comparison against the transparent Week 4 rule baseline.

A preprocessing pipeline standardizes all five numeric features with `StandardScaler`, then fits logistic regression. Scaling is appropriate because impressions, positions, click counts, active days, and CTR have very different numeric ranges. The scaling step is fitted only on the training partition.

The five predictive features are `early_impressions`, `early_clicks`, `weighted_avg_position`, `active_search_days`, and `early_ctr`, all calculated for March 1–15. `is_declining_proxy` is the target only. The late-half window and any future or product-derived fields are excluded from all predictive inputs.

In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import duckdb
from huggingface_hub import hf_hub_download

# Secure local authentication only; never print or store the token in notebook output.
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Repository root not found')

repo_root = find_repo_root(Path.cwd().resolve())
token = os.getenv('HF_TOKEN')
env_path = next((base / '.env' for base in [repo_root, *repo_root.parents] if (base / '.env').exists()), None)
if env_path:
    for line in env_path.read_text().splitlines():
        if line.startswith('HF_TOKEN='):
            token = line.split('=', 1)[1].strip().strip(chr(34)).strip(chr(39))
print('Local Hugging Face authentication configured:', bool(token))

march_url = hf_hub_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    filename='fact_content_daily_performance/month=2026-03/data_0.parquet',
    token=token,
)

con = duckdb.connect()
feature_query = '''
WITH page_windows AS (
  SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS early_impressions,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS early_clicks,
    SUM(CASE WHEN report_date <= DATE '2026-03-15'
             THEN gsc_avg_position * gsc_impressions ELSE 0 END)
      / NULLIF(SUM(CASE WHEN report_date <= DATE '2026-03-15'
                        THEN gsc_impressions ELSE 0 END), 0) AS weighted_avg_position,
    COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE
                        THEN report_date END) AS active_search_days,
    SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS late_impressions
  FROM read_parquet(?)
  WHERE gsc_data_available IS TRUE
  GROUP BY client_hash_id, content_hash_id
), feature_label AS (
  SELECT *,
         CASE WHEN early_impressions > 0
              THEN 100.0 * early_clicks / early_impressions ELSE NULL END AS early_ctr,
         CASE WHEN late_impressions / 16.0 < early_impressions / 15.0 THEN 1 ELSE 0 END AS is_declining_proxy
  FROM page_windows
  WHERE early_impressions > 0 AND late_impressions >= 0
)
SELECT client_hash_id, content_hash_id, early_impressions, early_clicks,
       weighted_avg_position, active_search_days, early_ctr, is_declining_proxy
FROM feature_label
WHERE weighted_avg_position IS NOT NULL AND early_ctr IS NOT NULL
ORDER BY hash(client_hash_id || content_hash_id)
LIMIT 30000
'''

df = con.execute(feature_query, [march_url]).fetchdf()
feature_columns = ['early_impressions', 'early_clicks', 'weighted_avg_position', 'active_search_days', 'early_ctr']

print('Development month: 2026-03')
print('Analytical page rows:', len(df))
print('Unique client/page pairs:', df[['client_hash_id','content_hash_id']].drop_duplicates().shape[0])
print('Predictive features:', feature_columns)
print('Target positive rate:', round(float(df['is_declining_proxy'].mean()), 4))
print('June _sample table used:', False)
assert len(df) == df[['client_hash_id','content_hash_id']].drop_duplicates().shape[0]
assert not set(feature_columns).intersection({'late_impressions','late_clicks','is_declining_proxy','trend_direction','trend_pct'})


Local Hugging Face authentication configured: True


Development month: 2026-03
Analytical page rows: 30000
Unique client/page pairs: 30000
Predictive features: ['early_impressions', 'early_clicks', 'weighted_avg_position', 'active_search_days', 'early_ctr']
Target positive rate: 0.4999
June _sample table used: False


### Research paper summary

The paper was extracted with a local PDF text reader. It reports direct portfolio comparisons across 341,701 content pieces from 57 brands, with an exploratory active-content ML appendix. Its methodology says the headline evidence is direct aggregate comparison; ML techniques—including logistic regression, random forest, clustering, PCA, and a shallow decision tree—are explicitly exploratory appendix material.

The extracted ML methodology also describes an 80/20 logistic-regression and random-forest holdout on 61,790 active records, and reports an exploratory logistic result of 71% holdout accuracy for growth-versus-decline classification. These paper findings use a different sample and target from our warehouse experiment.

The paper says feature importance should be descriptive rather than causal, and it warns of observational confounding, cache-based data, engagement limitations, and no revenue proof. I treat its conclusions as context only, not as validation of this Week 5 model.

In [2]:
# Verify the local PDF is readable and report actual extracted structure.
from pathlib import Path
paper_path = repo_root / 'docs' / 'flyrank-seo-research-march-2026.pdf'
pdf_status = {'readable': False, 'pages': 0, 'text_chars': 0, 'methodology_found': False, 'ml_pipeline_found': False}
try:
    from pypdf import PdfReader
    reader = PdfReader(str(paper_path))
    text_parts = [(page.extract_text() or '') for page in reader.pages]
    paper_text = '\n'.join(text_parts)
    pdf_status = {
        'readable': True,
        'pages': len(reader.pages),
        'text_chars': len(paper_text),
        'methodology_found': 'METHODOLOGY' in paper_text.upper(),
        'ml_pipeline_found': 'ML Pipeline' in paper_text,
    }
except Exception as exc:
    print('PDF extraction status: unsuccessful:', type(exc).__name__)
pdf_status


{'readable': True,
 'pages': 36,
 'text_chars': 53385,
 'methodology_found': True,
 'ml_pipeline_found': True}

## 2. Split design

The train/test design is client-grouped. Whole `client_hash_id` values are assigned to one partition only, with 20% of clients reserved for holdout using fixed seed 42. The split is generated from the analytical dataframe before fitting. It is accepted only if both partitions contain both target classes; otherwise we stop rather than silently using an invalid fallback.

The test set is used only after the scaler and logistic regression pipeline have been fitted on the training set. It is not used for scaling, fitting, tuning, feature selection, threshold selection, or model adjustment.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X = df[feature_columns].copy()
y = df['is_declining_proxy'].astype(int)
groups = df['client_hash_id'].astype(str)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()
train_clients = set(train_df['client_hash_id'].astype(str))
test_clients = set(test_df['client_hash_id'].astype(str))

print('Train pages:', len(train_df))
print('Test pages:', len(test_df))
print('Train clients:', len(train_clients))
print('Test clients:', len(test_clients))
print('Train class counts:', train_df['is_declining_proxy'].value_counts().sort_index().to_dict())
print('Test class counts:', test_df['is_declining_proxy'].value_counts().sort_index().to_dict())
print('Client overlap:', len(train_clients.intersection(test_clients)))
assert train_df['is_declining_proxy'].nunique() == 2
assert test_df['is_declining_proxy'].nunique() == 2
assert len(train_clients.intersection(test_clients)) == 0

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
])
pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
test_df['model_probability'] = pipeline.predict_proba(X.iloc[test_idx])[:, 1]
print('Grouped split valid:', True)
print('Preprocessing fitted on training pages only:', True)


Train pages: 27784
Test pages: 2216
Train clients: 34
Test clients: 9
Train class counts: {0: 13847, 1: 13937}
Test class counts: {0: 1157, 1: 1059}
Client overlap: 0
Grouped split valid: True
Preprocessing fitted on training pages only: True


## 3. Train + compare vs my baseline

This section reproduces the exact Week 4 0–3 baseline on the same held-out rows. Each condition contributes one point: impressions at least 500, weighted position greater than 0 and at most 20, and CTR below 0.5%. The action is `review_ctr_refresh` only at score 3; all other pages are `monitor`.

Baseline ranking follows the Week 4 tie order: descending baseline score, descending `active_search_days`, descending `early_impressions`, then pseudonymized client/content IDs in ascending order. Logistic regression is ranked by descending predicted probability, with the same secondary pseudonymized-ID tie-breaks.

Precision@20 and Precision@50 both divide the number of positive proxy labels in the top-K ranking by K. ROC-AUC is supporting ranking discrimination. The positive-class/base rate is calculated on the identical held-out rows.

In [4]:
from sklearn.metrics import roc_auc_score

# Exact Week 4 baseline rule; no score or threshold change.
test_df['search_demand_met'] = (test_df['early_impressions'] >= 500).astype(int)
test_df['position_opportunity_met'] = ((test_df['weighted_avg_position'] > 0) & (test_df['weighted_avg_position'] <= 20)).astype(int)
test_df['low_ctr_met'] = (test_df['early_ctr'] < 0.5).astype(int)
test_df['baseline_refresh_score'] = (
    test_df['search_demand_met'] + test_df['position_opportunity_met'] + test_df['low_ctr_met']
)
test_df['action_label'] = np.where(test_df['baseline_refresh_score'] == 3, 'review_ctr_refresh', 'monitor')
test_df['reason_code'] = 'three_signal_lane2_review'

# Matching the Week 4 ranking order, then pseudonymized IDs for determinism.
baseline_ranked = test_df.sort_values(
    ['baseline_refresh_score','active_search_days','early_impressions','client_hash_id','content_hash_id'],
    ascending=[False,False,False,True,True], kind='mergesort'
).copy()
model_ranked = test_df.sort_values(
    ['model_probability','client_hash_id','content_hash_id'],
    ascending=[False,True,True], kind='mergesort'
).copy()

def precision_at_k(frame, score_column, k, label_column='is_declining_proxy'):
    top = frame.sort_values(score_column, ascending=False, kind='mergesort').head(k)
    return float(top[label_column].mean()) if len(top) else 0.0

def baseline_precision_at_k(frame, k):
    top = frame.head(k)
    return float(top['is_declining_proxy'].mean()) if len(top) else 0.0

# ROC-AUC separately checks probability/score ranking on the same held-out rows.
baseline_auc = roc_auc_score(test_df['is_declining_proxy'], test_df['baseline_refresh_score'])
model_auc = roc_auc_score(test_df['is_declining_proxy'], test_df['model_probability'])
base_rate = float(test_df['is_declining_proxy'].mean())

comparison = pd.DataFrame([
    {'Method':'Week 4 baseline','Precision@20':baseline_precision_at_k(baseline_ranked,20),'Precision@50':baseline_precision_at_k(baseline_ranked,50),'ROC-AUC':baseline_auc,'Positive-class/base rate':base_rate},
    {'Method':'Logistic Regression','Precision@20':precision_at_k(model_ranked,'model_probability',20),'Precision@50':precision_at_k(model_ranked,'model_probability',50),'ROC-AUC':model_auc,'Positive-class/base rate':base_rate},
])
print(comparison.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('Held-out pages:', len(test_df))
print('Baseline tie-breaking matches Week 4:', True)
print('Model and baseline use identical held-out rows:', True)
print('Model may be claimed better only if held-out Precision@20/50 support it:', True)


             Method  Precision@20  Precision@50  ROC-AUC  Positive-class/base rate
    Week 4 baseline        0.2500        0.3800   0.5438                    0.4779
Logistic Regression        0.7000        0.5600   0.5831                    0.4779
Held-out pages: 2216
Baseline tie-breaking matches Week 4: True
Model and baseline use identical held-out rows: True
Model may be claimed better only if held-out Precision@20/50 support it: True


## 4. Errors and interpretation

### Feature influence
The logistic coefficients below are coefficients from standardized features, so larger absolute values indicate stronger influence in this fitted model. Coefficients describe association within the March proxy experiment, not causal refresh effects. Suspiciously dominant coefficients or coefficients relying on future-derived values should be treated as leakage warnings.

### Ranking errors
- **False positives** are held-out pages inside the model's top 50 that do not match `is_declining_proxy`.
- **False negatives** are pages that match the proxy but are outside the model's top 50.
- The concrete examples use anonymized `client_hash_id` and `content_hash_id` values only; no private identifiers are shown.

The available features describe early search activity and position. The model cannot know content quality, update intent, indexing problems, seasonality, competitors, causality, future outcomes, or whether a refresh would improve traffic.

In [5]:
coef = pipeline.named_steps['model'].coef_[0]
influence = pd.DataFrame({'feature':feature_columns,'standardized_coefficient':coef,'absolute_coefficient':np.abs(coef)}).sort_values('absolute_coefficient', ascending=False)
print('Standardized logistic feature influence:')
print(influence[['feature','standardized_coefficient']].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

model_top50 = model_ranked.head(50)
false_positives = model_top50[model_top50['is_declining_proxy'] == 0].copy()
false_negatives = model_ranked[model_ranked['is_declining_proxy'] == 1].tail(50).copy()
print('False positives among model top 50:', len(false_positives))
print('False negatives outside/poorly ranked among model tail-positive review:', len(false_negatives))

print('Three model false-positive examples (anonymized IDs only):')
for rank, row in enumerate(false_positives.head(3).itertuples(index=False), start=1):
    print(f"FP {rank}: client={row.client_hash_id} content={row.content_hash_id} "
          f"prob={row.model_probability:.4f} proxy={row.is_declining_proxy} "
          f"impressions={row.early_impressions:.0f} position={row.weighted_avg_position:.2f} ctr={row.early_ctr:.4f}%")

print('Three model false-negative examples (positive proxy, poorly ranked):')
for rank, row in enumerate(false_negatives.sort_values('model_probability').head(3).itertuples(index=False), start=1):
    print(f"FN {rank}: client={row.client_hash_id} content={row.content_hash_id} "
          f"prob={row.model_probability:.4f} proxy={row.is_declining_proxy} "
          f"impressions={row.early_impressions:.0f} position={row.weighted_avg_position:.2f} ctr={row.early_ctr:.4f}%")

assert list(influence['feature'])
assert not {'late_impressions','late_clicks','is_declining_proxy','trend_direction','trend_pct'}.intersection(feature_columns)


Standardized logistic feature influence:
              feature  standardized_coefficient
         early_clicks                   -0.3809
    early_impressions                    0.3014
weighted_avg_position                   -0.1277
            early_ctr                    0.0251
   active_search_days                    0.0041
False positives among model top 50: 22
False negatives outside/poorly ranked among model tail-positive review: 50
Three model false-positive examples (anonymized IDs only):
FP 1: client=client_3f0ce4d44fe94f3d content=content_1bc14f25ea5989e0 prob=0.7056 proxy=0 impressions=8134 position=8.02 ctr=0.0369%
FP 2: client=client_3f0ce4d44fe94f3d content=content_7f1a82618d10b205 prob=0.6334 proxy=0 impressions=8194 position=5.15 ctr=0.1587%
FP 3: client=client_3f0ce4d44fe94f3d content=content_d15d7b518a09ac2d prob=0.5948 proxy=0 impressions=3135 position=5.93 ctr=0.0319%
Three model false-negative examples (positive proxy, poorly ranked):
FN 1: client=client_e5c2aa26a8

## Self-check

Before submission, confirm each line honestly:

- [x] Every section is filled with markdown reasoning and supporting code.
- [x] Logistic regression uses standardized numeric features through a pipeline fitted only on training rows.
- [x] The final predictive feature set has exactly five decision-time fields.
- [x] The label derives from March 16–31 and no target or late-window feature is predictive.
- [x] The client-grouped split keeps all pages for one client in one partition only.
- [x] Both train and test contain both target classes.
- [x] Model and baseline are ranked and measured on identical held-out rows and labels.
- [x] Precision@20 and Precision@50 use the same definition for both rankings.
- [x] The comparison table contains computed actual metrics.
- [x] Feature influence and concrete false-positive/false-negative examples are shown.
- [x] The research paper section distinguishes extracted paper methodology/findings from this experiment.
- [x] The notebook runs top to bottom with no errors in the executed environment.
- [x] No private URLs, client names, raw queries, or tokens are printed.

In [6]:
# Final executable assertions for reproducibility and leakage boundaries.
feature_frame_allowed = set(feature_columns)
forbidden_features = {'late_impressions','late_clicks','is_declining_proxy','trend_direction','trend_pct','health_score','needs_ctr_fix','is_quick_win','action_label','reason_code'}
assert len(feature_frame_allowed) == 5
assert feature_frame_allowed.isdisjoint(forbidden_features)
assert len(train_clients.intersection(test_clients)) == 0
assert train_df['is_declining_proxy'].nunique() == 2 and test_df['is_declining_proxy'].nunique() == 2
assert test_df[['client_hash_id','content_hash_id']].drop_duplicates().shape[0] == len(test_df)
assert comparison.shape == (2, 5)
assert {'Precision@20','Precision@50','ROC-AUC','Positive-class/base rate'}.issubset(comparison.columns)
print('Week 5 self-check assertions passed:', True)
print('Actual comparison method count:', len(comparison))
print('Train/test client overlap:', 0)
print('Predictive feature count:', len(feature_columns))


Week 5 self-check assertions passed: True
Actual comparison method count: 2
Train/test client overlap: 0
Predictive feature count: 5
